In [1]:
from kiblib.utils.db import DbConn
import pandas as pd

In [2]:
db_conn = DbConn().create_engine()

In [3]:
query = """SELECT s.biblionumber, s.librarian, s.closed, b.title 
FROM koha_prod.subscription s 
JOIN biblio b on b.biblionumber = s.biblionumber
#WHERE s.closed = 1"""

In [4]:
query_2 = """SELECT biblionumber,barcode FROM koha_prod.items"""

In [5]:
items = pd.read_sql(query_2,db_conn)

In [6]:
items = items.groupby('biblionumber')['barcode'].count().to_frame()

In [7]:
subscriptions = pd.read_sql(query,db_conn)

In [8]:
subscriptions.loc[subscriptions['closed']==0,'a/ ouvert'] = 1
subscriptions.loc[subscriptions['closed']==1,'b/ fermé'] = 1
subscriptions['nb'] = 1

In [9]:
# Nb de lignes avant le mergem 
len (f'nb notices avant le merge {subscriptions}')

1624

In [10]:
subscriptions = subscriptions.merge(items,left_on='biblionumber',
                                    right_on='biblionumber',
                                    how='left'
                                   )

# Nb de lignes après merge
len (f'nb notices après le merge {subscriptions}')

1732

In [11]:
subscriptions.loc[subscriptions['barcode'].notna(),'exemplaire'] = 1

In [12]:
subscriptions_details = subscriptions.pivot_table(index='title',
                                                  values=['a/ ouvert','b/ fermé','nb','barcode'],
                                                  aggfunc={'a/ ouvert':'sum',
                                                           'b/ fermé':'sum',
                                                           'nb':'sum',
                                                           'barcode':sum,
                                                          },
                                                  fill_value=0,
                                                  #sort=True,
                                                  observed=True
                                                 )
cols_renamed = {"nb":"nb_total_abonnement","barcode":"Nb exemplaires total rattaché à cette notice"}
cols_order = ['a/ ouvert','b/ fermé','nb_total_abonnement','Nb exemplaires total rattaché à cette notice']
subscriptions_details.rename(columns=cols_renamed,inplace=True)
subscriptions_details = subscriptions_details[cols_order]

In [13]:
sub_totally_closed = subscriptions_details[subscriptions_details['a/ ouvert']==0]
sub_totally_closed

,a/ ouvert,b/ fermé,nb_total_abonnement,Nb exemplaires total rattaché à cette notice
title,,,,
Abus dangereux,0,1,1,2
Afrique Asie,0,1,1,0
Alternatives internationales,0,1,1,4
Baïka,0,1,1,9
Brèves,0,1,1,11
Cahiers pédagogiques,0,1,1,17
Canard PC hardware,0,1,1,6
Causette,0,1,1,6
Comment ça marche,0,1,1,4


In [14]:
sub_totally_closed_WhithoutItems = sub_totally_closed[sub_totally_closed['Nb exemplaires total rattaché à cette notice']==0]

In [15]:
sub_totally_closed_WhithoutItems

,a/ ouvert,b/ fermé,nb_total_abonnement,Nb exemplaires total rattaché à cette notice
title,,,,
Afrique Asie,0,1,1,0
Environnement magazine,0,1,1,0
Eureka,0,1,1,0
Les clés de l'actualité junior,0,1,1,0
Netsources,0,1,1,0
Quelle histoire magazine,0,1,1,0
Séries mag,0,1,1,0
Top santé,0,1,1,0


In [16]:
len(sub_totally_closed)

48

In [17]:
sub_totally_closed.to_excel('/home/kibini/listing_PeriosAbonnementsTotalementFermés.xlsx')
sub_totally_closed_WhithoutItems.to_excel('/home/kibini/listing_PeriosAbonnementsTotalementFermés_SansNoticesRattachées.xlsx')